# 🛰️ Satellite Fine-Tuning (TinyUNetSiam quick-train) — One-click Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/satellite_finetune_colab.ipynb)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Task](https://img.shields.io/badge/task-cd-4c8cbf) ![Auto](https://img.shields.io/badge/runs-automatically-3fb950)

Fine-tune a small siamese change-detection network **on your own pre/post pairs** — or on bundled synthetic satellite scenes — in ~3 minutes on a free CPU runtime. Produces a real trained checkpoint `tiny_unet_cd.pth` (mode **QUICK-TRAIN**), ready to upload into `change_detection_colab.ipynb` for CHECKPOINT-mode inference. Scale up to BIT / ChangeMamba with the repo `train.py` when you're ready.

| ▶️ How to run | |
|---|---|
| **Everything automatic** | Click **Runtime ▸ Run all** (Ctrl+F9): setup → demo scene → inference → dashboard → export chain without any further clicks |
| **With your data** | Run the cells once, then click the 📤 upload buttons — the pipeline re-runs automatically after upload |
| **No image at hand?** | The 🛟 fallback generates a clearly-labeled synthetic flood scene — the notebook *never blocks or crashes* |

**📤 Upload preview — what to have ready:**

| file | what it is | typical size |
|---|---|---|
| **`dataset.zip`** | pairs named <id>_pre.png / <id>_post.png / <id>_mask.png | 128–1024 px tiles, ≤ 200 MB |
| **`no zip?`** | auto-generates 48 synthetic training pairs | — |

> 🔖 Every result is stamped with its mode: **CHECKPOINT** (your `.pth`), **QUICK-TRAIN** (genuinely briefly-trained), or **SYNTHETIC-DEMO** (clearly-labeled heuristic). Nothing is faked.

In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent, restart-safe {display-mode:"form"}
AUTO_RUN = True  #@param {type:"boolean"}
SHOW_PREVIEW_ON_UPLOAD = True  #@param {type:"boolean"}
import os, sys, glob, json, math, random, subprocess, traceback, datetime, textwrap

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

colab_kit.pip_install("ipywidgets", "pillow", "numpy", "imageio", "scikit-image", "tifffile", "matplotlib")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageDraw
import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output
from colab_kit import (make_synthetic, infer, overlay, heatmap, to_u8, stats_for,
                       stylize_poster, read_any, verdict, show_pair, run_inference,
                       run_dashboard, export_results_zip_a, nb_save_upload, nb_apply,
                       nb_upload_pre, nb_upload_post, nb_use_synthetic,
                       _field, _fbm, _stretch, _blur, _median_mask, _terrain, _hsv,
                       segment_heuristic, change_heuristic, _otsu)

random.seed(42); np.random.seed(42)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
DOMAIN = "flood"          # this notebook's disaster domain
TASK   = "cd"            # "seg" | "cd" | "both"
GSD_M  = 10.0               # ground sample distance (m/pixel) for km² stats
NOTEBOOK = "satellite_finetune_colab.ipynb"
STATE = {"domain": DOMAIN, "task": "seg" if TASK != "cd" else "cd",
         "pre": None, "post": None, "gt": None, "mask": None, "source": "none"}
print("✅ Environment ready — notebook:", NOTEBOOK, "| domain:", DOMAIN, "| task:", TASK)

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — buttons + inline preview, never blocks {display-mode:"form"}
# upload logic lives in colab_kit (single source of truth, unit-tested in CI)
def _save_upload(name, data):
    return colab_kit.nb_save_upload(globals(), name, data)

def _apply(kind, arr):
    return colab_kit.nb_apply(globals(), kind, arr)

def upload_pre(_=None):
    return colab_kit.nb_upload_pre(globals())

def upload_post(_=None):
    return colab_kit.nb_upload_post(globals())

def use_synthetic(_=None):
    return colab_kit.nb_use_synthetic(globals())


SPEC = [['dataset.zip',
  'pairs named <id>_pre.png / <id>_post.png / <id>_mask.png',
  '128–1024 px tiles, ≤ 200 MB'],
 ['no zip?', 'auto-generates 48 synthetic training pairs', '—']]
W = 70
def print_spec():
    print("┌" + "─" * W + "┐")
    for line in textwrap.wrap("📤 WHAT TO UPLOAD — " + NOTEBOOK, W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    for label, desc, size in SPEC:
        wrapped = textwrap.wrap(label + " — " + desc + "  (" + size + ")", W - 6)
        for j, line in enumerate(wrapped):
            print(("│   • " if j == 0 else "│     ") + line.ljust(W - 6) + "│")
    for line in textwrap.wrap(
            "accepted: .png .jpg .jpeg .tif .tiff .h5 | sweet spot: 512–2048 px, ≤ 20 MB each | "
            "GeoTIFF georeference is kept when present", W - 2):
        print("│ " + line.ljust(W - 2) + "│")
    print("└" + "─" * W + "┘")
print_spec()

out_up = widgets.Output(layout={"border": "1px solid #ddd"})
domain_sel = widgets.Dropdown(
    options=["flood", "wildfire", "tsunami", "landslide", "volcano"],
    value="flood", description="Train domain:")
def _sync_dom(*_):
    STATE["domain"] = domain_sel.value
domain_sel.observe(_sync_dom, names="value")
# (task fixed for this notebook)
gsd_box = widgets.FloatText(value=GSD_M, description="GSD m/px:",
                            tooltip="ground sample distance, used for km² statistics")
btn_post = widgets.Button(description="⬆️ Upload POST image", button_style="success", icon="upload")
btn_extra = None

btn_extra = widgets.Button(description="⬆️ Upload dataset .zip (id_pre/id_post/id_mask)",
                           button_style="info", icon="upload")
def _scan_pairs_local(root):
    ids = sorted({os.path.basename(p).rsplit("_", 1)[0]
                  for p in glob.glob(os.path.join(root, "**", "*_pre.*"), recursive=True)})
    out = []
    for i in ids:
        pre = glob.glob(os.path.join(root, "**", i + "_pre.*"), recursive=True)
        post = glob.glob(os.path.join(root, "**", i + "_post.*"), recursive=True)
        msk = (glob.glob(os.path.join(root, "**", i + "_mask.*"), recursive=True)
               or glob.glob(os.path.join(root, "**", i + "_label.*"), recursive=True))
        if pre and post and msk:
            out.append((pre[0], post[0], msk[0]))
    return out
def upload_dataset(_=None):
    with out_up:
        clear_output()
        print("📂 choose a .zip with <id>_pre / <id>_post / <id>_mask images …")
    try:
        import zipfile as zf, io
        for name, data in _gfiles.upload().items():
            zf.ZipFile(io.BytesIO(data)).extractall("data_upload")
            pairs = _scan_pairs_local("data_upload")
            with out_up:
                print("✅ found", len(pairs), "labeled pairs")
            if pairs:
                STATE["pairs"] = pairs
                STATE["source"] = "user"
                if AUTO_RUN and "quick_train" in globals():
                    quick_train()
    except Exception as e:
        with out_up:
            print("upload failed:", e)
btn_extra.on_click(upload_dataset)


btn_pre = widgets.Button(description="⬆️ Upload PRE image", button_style="primary", icon="upload")
btn_synth = widgets.Button(description="🛟 Use bundled synthetic sample",
                           button_style="warning", icon="picture")
btns = [btn_pre]
if TASK in ("cd", "both"):
    btns.insert(1, btn_post)
if btn_extra is not None:
    btns.append(btn_extra)
box = widgets.VBox([widgets.HBox(btns), widgets.HBox([btn_synth, gsd_box]), out_up])






btn_pre.on_click(upload_pre)
if TASK in ("cd", "both"):
    btn_post.on_click(upload_post)
btn_synth.on_click(use_synthetic)
display(box)

if AUTO_RUN and STATE["pre"] is None:
    print("⚡ AUTO_RUN: generating bundled synthetic sample — upload later to re-run automatically")
    use_synthetic()

In [ ]:
#@title 🏋️ TRAIN — TinyUNetSiam quick-training (CPU-friendly) {display-mode:"form"}
EPOCHS = 3  #@param {type:"integer"}
N_SYNTHETIC = 48  #@param {type:"integer"}
BATCH = 8  #@param {type:"integer"}

import torch
import torch.nn as nn

class TinyUNetSiam(nn.Module):
    """6-channel (pre+post) -> 1-channel change logits. 3-level UNet, CPU-friendly."""
    def __init__(self, ch=16):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.ReLU())
        self.e1 = blk(6, ch);    self.p1 = nn.MaxPool2d(2)
        self.e2 = blk(ch, ch*2); self.p2 = nn.MaxPool2d(2)
        self.mid = blk(ch*2, ch*4)
        self.u2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2)
        self.d2 = blk(ch*4, ch*2)
        self.u1 = nn.ConvTranspose2d(ch*2, ch, 2, stride=2)
        self.d1 = blk(ch*2, ch)
        self.out = nn.Conv2d(ch, 1, 1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        m  = self.mid(self.p2(e2))
        d2 = self.d2(torch.cat([self.u2(m), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

def _tile(a, size=128):
    im = Image.fromarray(to_u8(a)).resize((size, size))
    return np.asarray(im, np.float32) / 255.0
def _load_pair(pre_p, post_p, msk_p, size=128):
    pre, post = read_any(pre_p), read_any(post_p)
    m = read_any(msk_p).mean(-1) > 0.5
    return _tile(pre, size), _tile(post, size), _tile(m[..., None], size)[..., 0].astype(np.float32)
def _dataset():
    if STATE.get("pairs"):
        return [_load_pair(*t) for t in STATE["pairs"][:200]]
    out = []
    for _k in range(N_SYNTHETIC):
        pre, post, gt = make_synthetic(STATE["domain"], 128, 128, seed=random.randint(0, 9999))
        out.append((pre, post, gt.astype(np.float32)))
    return out
def quick_train():
    data = _dataset()
    n_val = max(2, len(data) // 5)
    tr, va = data[:-n_val], data[-n_val:]
    net = TinyUNetSiam()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    lossf = torch.nn.BCEWithLogitsLoss()
    print("training pairs:", len(tr), "| val:", len(va), "| device: cpu")
    for ep in range(EPOCHS):
        random.shuffle(tr)
        net.train(); tot = 0.0
        for i in range(0, len(tr) - BATCH + 1, BATCH):
            chunk = tr[i:i + BATCH]
            pre = np.stack([c[0] for c in chunk])
            post = np.stack([c[1] for c in chunk])
            msk = np.stack([c[2] for c in chunk])
            x = torch.from_numpy(np.concatenate([pre.transpose(0, 3, 1, 2),
                                                 post.transpose(0, 3, 1, 2)], 1))  # N,6,H,W
            y = torch.from_numpy(msk).unsqueeze(1)                    # N,1,H,W
            if random.random() < 0.5:
                x, y = torch.flip(x, [3]), torch.flip(y, [3])
            opt.zero_grad()
            loss = lossf(net(x), y)
            loss.backward(); opt.step()
            tot += float(loss) * len(chunk)
        net.eval(); inter = union = 0
        with torch.no_grad():
            for pre, post, msk in va:
                x = torch.from_numpy(np.concatenate([pre.transpose(2, 0, 1)[None],
                                                     post.transpose(2, 0, 1)[None]], 1))
                logits = net(x)[0, 0].numpy() > 0.0
                inter += int((logits & (msk > 0.5)).sum())
                union += int((logits | (msk > 0.5)).sum())
        iou = inter / max(1, union)
        print(f"epoch {ep+1}/{EPOCHS} | loss {tot/len(tr):.4f} | val IoU {iou:.3f}")
    ck = os.path.join(OUT_DIR, "tiny_unet_cd.pth")
    torch.save(net.state_dict(), ck)
    STATE["net"] = net
    STATE["ckpt"] = ck
    print("💾 saved", ck, "— mode QUICK-TRAIN (real training, small net)")
    if AUTO_RUN and "run_eval" in globals():
        run_eval()
quick_train() if AUTO_RUN else print("call quick_train() after uploading (or set AUTO_RUN)")

In [ ]:
#@title ▶️ EVAL — fresh scene through the trained net + dashboard {display-mode:"form"}
MODE_TAG = "QUICK-TRAIN"
def run_eval():
    STATE["mode_tag"] = MODE_TAG
    net = STATE.get("net")
    if net is None:
        print("no trained net in STATE — the TRAIN cell runs first under AUTO_RUN")
        return
    pre, post, gt = make_synthetic(STATE["domain"], 256, 256, seed=random.randint(0, 9999))
    x = torch.from_numpy(np.concatenate([pre.transpose(2, 0, 1), post.transpose(2, 0, 1)])[None])
    with torch.no_grad():
        logits = net(x)[0, 0].numpy()
    mask = _median_mask(logits > 0.0, 5)
    iou = float((mask & gt).sum()) / max(1, float((mask | gt).sum()))
    fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
    for a, im, t in zip(ax, [pre, post, heatmap(mask), overlay(post, mask)],
                        ["PRE", "POST", "logit heatmap", "overlay — QUICK-TRAIN"]):
        a.imshow(np.clip(im, 0, 1)); a.set_title(t, fontsize=10); a.axis("off")
    fig.suptitle(f"TinyUNetSiam quick-train eval | val-scene IoU {iou:.3f}", fontweight="bold")
    ev = os.path.join(OUT_DIR, "finetune_eval.png")
    fig.savefig(ev, dpi=110, bbox_inches="tight"); plt.show()
    print("📊 saved", ev, "| mode QUICK-TRAIN")
    if AUTO_RUN and "run_export" in globals():
        run_export()
run_eval() if AUTO_RUN else print("call run_eval() after training")

In [ ]:
#@title 📦 EXPORT — results zip + optional Drive copy + auto-download {{display-mode:"form"}}
USE_DRIVE = False  #@param {{type:"boolean"}}

def run_export():
    return colab_kit.export_results_zip_a(globals(), use_drive=USE_DRIVE)

run_export() if AUTO_RUN else print("call run_export() when ready")

## 🚀 Scale up to the real models

| step | how |
|---|---|
| Use the checkpoint now | upload `outputs/tiny_unet_cd.pth` into **`change_detection_colab.ipynb`** → CHECKPOINT mode |
| Real benchmarks | `python train.py --cfg configs/<domain>/<dataset>/<model>.yaml` (see README → Quick Start) |
| Bigger models | BIT, ChangeMamba (ChangeOS), SpectralGPT+ under `src/models/` |
| Per-domain data prep | `scripts/data_prep/<domain>/README.md` |
| Honest labeling | results from this net are stamped **QUICK-TRAIN**, never presented as SOTA |